# SETI RF GPU Analysis Engine
This notebook streams public observation files directly from Breakthrough Listen's AWS S3 bucket and runs a GPU-accelerated Doppler drift search using `turboSETI`.

In [ ]:
# --- CELL 1: Install CUDA & SETI dependencies ---
!pip install -q blimpy turbo_seti cupy-cuda12x fsspec s3fs

In [ ]:
# --- CELL 2: Verify GPU availability ---
import cupy as cp
print("[✓] GPU Device:", cp.cuda.runtime.getDeviceProperties(0)['name'].decode('utf-8'))

In [ ]:
# --- CELL 3: Run GPU Doppler Search across Public S3 Stream ---
from turbo_seti.find_doppler.find_doppler import FindDoppler
import os

# Public AWS S3 URL (Breakthrough Listen Voyager 1 observation)
s3_target = "https://breakthrough.s3.amazonaws.com/voyager/Voyager1.single_coarse.fine_res.h5"

os.makedirs("./output", exist_ok=True)

# Initialize Doppler Search Engine
fdop = FindDoppler(
    datafile=s3_target,
    max_drift=4.0,  # Search for signals drifting -4.0 to +4.0 Hz/s
    snr=10.0,       # Signal-to-Noise Ratio threshold
    out_dir="./output"
)

# Execute GPU-accelerated search
print("[+] Launching GPU-accelerated de-doppler search...")
fdop.search(use_gpu=True)
print("[✓] Search completed! Results written to ./output/")

In [ ]:
# --- CELL 4: Inspect Generated Candidate Hits ---
import glob

dat_files = glob.glob("./output/*.dat")
if dat_files:
    with open(dat_files[0], 'r') as f:
        print("\n--- CANDIDATE HITS REPORT ---")
        print(f.read()[:2000]) # Display top hits
else:
    print("[-] No .dat output file found.")